# Sales Data — Exploratory Data Analysis

Quick look at the dataset before we start modeling.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')

In [ ]:
from src.data_preprocessing import preprocess_all_states
state_series = preprocess_all_states('../data/sales_data.xlsx')
print(f'States loaded: {len(state_series)}')

In [ ]:
# Look at a few states
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
sample_states = ['Texas', 'California', 'Florida', 'New York']

for ax, state in zip(axes.flatten(), sample_states):
    if state in state_series:
        df = state_series[state]
        ax.plot(df['date'], df['sales'], linewidth=1.5)
        ax.set_title(state)
        ax.set_ylabel('Sales ($)')
        ax.tick_params(axis='x', rotation=30)

plt.suptitle('Weekly Sales by State', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of sales across states
all_data = []
for state, df in state_series.items():
    tmp = df.copy()
    tmp['state'] = state
    all_data.append(tmp)

all_df = pd.concat(all_data)

state_avg = all_df.groupby('state')['sales'].mean().sort_values(ascending=True)
state_avg.plot(kind='barh', figsize=(12, 10), color='steelblue', alpha=0.8)
plt.title('Average Weekly Sales by State')
plt.xlabel('Average Sales ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Seasonality check — average sales by month
all_df['month'] = all_df['date'].dt.month
monthly_avg = all_df.groupby('month')['sales'].mean()

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
plt.figure(figsize=(10, 4))
plt.bar(range(1, 13), monthly_avg, color='coral', alpha=0.85)
plt.xticks(range(1, 13), month_names)
plt.title('Average Sales by Month (All States)')
plt.ylabel('Avg Sales ($)')
plt.tight_layout()
plt.show()
print('Seasonality is visible — higher in mid-year and around holidays')

In [ ]:
# Check stationarity for Texas
from statsmodels.tsa.stattools import adfuller

tx = state_series.get('Texas')
if tx is not None:
    result = adfuller(tx['sales'].dropna())
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    print('Stationary:', result[1] < 0.05)

In [ ]:
# Feature engineering preview
from src.feature_engineering import build_features

tx = state_series.get('Texas')
if tx is not None:
    feat_df = build_features(tx)
    print('Features built:', feat_df.shape)
    feat_df[['date', 'sales', 'lag_1', 'lag_4', 'rolling_mean_4', 'is_holiday']].tail()